Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo\\SNpollutionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [4]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(43800, 6)

In [10]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


Espacio de búsqueda

In [11]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [12]:
futuros = 24
pasados  = 12

In [13]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 0])


In [14]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43765, 12, 6)
Dimensiones de Y: (43765, 1)


In [15]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894 ]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347 ]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449]
 [ 0.57004095 -0.65625711 -1.34931411  0.90832835 -0.38094383 -0.09555657]]


Se dividen nuevamente los conjuntos de datos

In [16]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30635, 12, 6)
Las dimensiones de testX son:  (8797, 12, 6)
Las dimensiones de valX son:  (4333, 12, 6)


In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30635, 1)
Las dimensiones de testY son:  (8797, 1)
Las dimensiones de valY son:  (4333, 1)


Se crean métricas para medir desempeño

In [18]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [19]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [20]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=128,
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [21]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

240/240 - 14s - 57ms/step - ia: 0.1860 - loss: 1.0906 - mae: 0.8155 - rmse: 1.0399 - smape: 1.6083 - val_ia: 0.2529 - val_loss: 0.9710 - val_mae: 0.7438 - val_rmse: 0.8843 - val_smape: 1.9040

Epoch 2/128                                           

240/240 - 3s - 12ms/step - ia: 0.1706 - loss: 1.0407 - mae: 0.7674 - rmse: 1.0132 - smape: 1.6251 - val_ia: 0.2529 - val_loss: 0.9672 - val_mae: 0.7363 - val_rmse: 0.8782 - val_smape: 1.9842

Epoch 3/128                                           

240/240 - 3s - 12ms/step - ia: 0.1570 - loss: 1.0271 - mae: 0.7604 - rmse: 1.0071 - smape: 1.6438 - val_ia: 0.2529 - val_loss: 0.9663 - val_mae: 0.7362 - val_rmse: 0.8780 - val_smape: 1.9829

Epoch 4/128                                           

240/240 - 3s - 13ms/step - ia: 0.1416 - loss: 1.0194 - mae: 0.7571 - rmse: 1.0021 - smape: 1.6686 - val_ia: 0.2530 - val_loss: 0.9654 - val_mae: 0.7371 - val_rmse: 0.8785 - val_smape: 1.9735

Epoch 5

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                        

1915/1915 - 65s - 34ms/step - ia: 0.3622 - loss: 0.8322 - mae: 0.6725 - rmse: 0.8784 - smape: 1.3291 - val_ia: 0.2508 - val_loss: 0.7513 - val_mae: 0.6369 - val_rmse: 0.7046 - val_smape: 1.2473

Epoch 2/128                                                                        

1915/1915 - 80s - 42ms/step - ia: 0.4087 - loss: 0.7844 - mae: 0.6464 - rmse: 0.8515 - smape: 1.2439 - val_ia: 0.2495 - val_loss: 0.7510 - val_mae: 0.6396 - val_rmse: 0.7093 - val_smape: 1.2457

Epoch 3/128                                                                        

1915/1915 - 81s - 42ms/step - ia: 0.4238 - loss: 0.7656 - mae: 0.6361 - rmse: 0.8429 - smape: 1.2190 - val_ia: 0.2541 - val_loss: 0.7390 - val_mae: 0.6235 - val_rmse: 0.6931 - val_smape: 1.2129

Epoch 4/128                                                                        

1915/1915 - 83s - 43ms/step - ia: 0.4341 - loss: 0.7490 - mae: 0.6281 - rms

...

Epoch 1/128                                                                        

1915/1915 - 35s - 18ms/step - ia: 0.2649 - loss: 1.0355 - mae: 0.7633 - rmse: 0.9802 - smape: 1.4987 - val_ia: 0.2289 - val_loss: 0.8168 - val_mae: 0.6784 - val_rmse: 0.7395 - val_smape: 1.4504

Epoch 2/128                                                                        

1915/1915 - 26s - 14ms/step - ia: 0.3468 - loss: 0.8892 - mae: 0.7010 - rmse: 0.9097 - smape: 1.3451 - val_ia: 0.2349 - val_loss: 0.7960 - val_mae: 0.6743 - val_rmse: 0.7390 - val_smape: 1.3598

Epoch 3/128                                                                        

1915/1915 - 24s - 13ms/step - ia: 0.3627 - loss: 0.8581 - mae: 0.6862 - rmse: 0.8941 - smape: 1.3195 - val_ia: 0.2439 - val_loss: 0.7766 - val_mae: 0.6493 - val_rmse: 0.7155 - val_smape: 1.2839

Epoch 4/128                                                                        

1915/1915 - 25s - 13ms/step - ia: 0.3631 - loss: 0.8501 - mae: 0.6830 - rms

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



3830/3830 - 23s - 6ms/step - ia: 0.2329 - loss: 0.9875 - mae: 0.7433 - rmse: 0.9285 - smape: 1.6850 - val_ia: 0.1902 - val_loss: 0.9362 - val_mae: 0.7112 - val_rmse: 0.7474 - val_smape: 1.6286

Epoch 2/128                                                                         

3830/3830 - 20s - 5ms/step - ia: 0.2324 - loss: 0.9847 - mae: 0.7421 - rmse: 0.9260 - smape: 1.6864 - val_ia: 0.1901 - val_loss: 0.9350 - val_mae: 0.7117 - val_rmse: 0.7478 - val_smape: 1.6361

Epoch 3/128                                                                         

3830/3830 - 19s - 5ms/step - ia: 0.2349 - loss: 0.9794 - mae: 0.7404 - rmse: 0.9252 - smape: 1.6842 - val_ia: 0.1901 - val_loss: 0.9339 - val_mae: 0.7121 - val_rmse: 0.7482 - val_smape: 1.6426

Epoch 4/128                                                                         

3830/3830 - 19s - 5ms/step - ia: 0.2358 - loss: 0.9776 - mae: 0.7400 - rmse: 0.9239 - smape: 1.6900 - val_ia: 0.1900 - val_loss: 0.9328 - val_mae: 0.7124 - val_

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                          

240/240 - 15s - 61ms/step - ia: 0.2847 - loss: 0.9418 - mae: 0.7229 - rmse: 0.9651 - smape: 1.4360 - val_ia: 0.3098 - val_loss: 0.7822 - val_mae: 0.6404 - val_rmse: 0.7871 - val_smape: 1.2760

Epoch 2/128                                                                          

240/240 - 4s - 15ms/step - ia: 0.3666 - loss: 0.8633 - mae: 0.6895 - rmse: 0.9232 - smape: 1.3197 - val_ia: 0.3118 - val_loss: 0.7770 - val_mae: 0.6569 - val_rmse: 0.8001 - val_smape: 1.3175

Epoch 3/128                                                                          

240/240 - 4s - 17ms/step - ia: 0.3746 - loss: 0.8514 - mae: 0.6822 - rmse: 0.9171 - smape: 1.3036 - val_ia: 0.3150 - val_loss: 0.7723 - val_mae: 0.6504 - val_rmse: 0.7952 - val_smape: 1.2945

Epoch 4/128                                                                          

240/240 - 4s - 15ms/step - ia: 0.3781 - loss: 0.8420 - mae: 0.6774 - rmse: 

In [22]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}
